# Social Media Donation Conversion — Predictive Model

---

## 1. Problem Framing

### Business Problem

Hearth Haven's primary donor acquisition channel is social media, but the
organization posts without a reliable way to know which content choices actually
lead to donations versus which just generate likes. The founders admit they are
not experienced with social media strategy, do not have a marketing team, and
currently make posting decisions based on intuition alone.

This pipeline answers the question: **Which post characteristics predict whether
a social media post will lead to at least one confirmed donation?**

The deployed output is a **Donation Conversion Score (0–100)** surfaced in the
social media management dashboard, giving staff a data-informed guide for content
planning — before they post.

### Who Cares About This

- **Organization leadership** — needs to maximize donation revenue from social media
  without a dedicated marketing budget.
- **Staff managing social media** — need concrete, actionable guidance on what to post,
  when, and on which platform.
- **Donor relations** — understanding which content types drive actual giving informs
  future fundraising strategy.

### Predictive vs. Explanatory

This pipeline uses a **predictive approach**. The goal is to score future post ideas
by their estimated donation conversion probability. Out-of-sample performance is the
primary criterion.

RandomForest feature importances (Section 3.5) reveal which post characteristics
the model relies on most — discussed in Section 5 as associations that are actionable
even without causal claims.

### Success Metrics

- **Primary:** ROC-AUC — essential for the 7.6% positive rate where accuracy alone
  is misleading
- **Secondary:** F1, Balanced Accuracy, Average Precision
- **Operational threshold:** tuned toward precision — false positives (predicting
  conversion on non-converting content) misdirect limited staff effort

### A Note on Training Data

RandomForest is especially susceptible to memorizing duplicated rows, producing
optimistic CV estimates when near-identical records appear across folds. The
pipeline architecture and feature relationships are valid; the AUC = 1.000 on the
test set reflects memorization from data duplication. Confidence in absolute score
values will increase as Hearth Haven accumulates independently collected post data.

---
## 2. Data Acquisition, Preparation & Exploration

In [43]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

from functions.fn_domain_prep import prepare_social_media
from functions.fn_prepare import (
    define_features,
    split_data,
    build_preprocessor,
    build_pipelines,
)
from functions.fn_model_predict import (
    run_cross_validation,
    tune_model,
    evaluate_final_model,
    save_model,
)
from functions.fn_model_causal import fit_causal_classification, get_coefficients

print("All imports successful.")

All imports successful.


### 2.1 Load and Prepare Data

`prepare_social_media()` encodes every cleaning and feature engineering decision from
`eda_social_media.ipynb`. It tries Azure SQL first and falls back to CSV automatically.

**Tables joined:** `social_media_posts`, `donations`

**Key preparation decisions encoded:**
- Structural columns dropped: `platform_post_id`, `post_url`, `caption`, `hashtags`
- Temporal features engineered: `post_month`, `post_is_weekend` from `created_at`
- Intentional nulls filled: video metrics → 0, `call_to_action_type` → 'None'
- `campaign_name` nulls → 'None' (meaningful signal — not linked to campaign)
- Rare campaign categories binned: GivingTuesday → 'Other' (< 5%)
- Outliers capped and skew transformed: `impressions`, `reach`
- `boost_budget_php` NOT transformed — 84% zeros collapse IQR fence; signal
  preserved via `is_boosted` boolean
- `led_to_donation` engineered: 1 if post led to ≥ 1 confirmed social donation

In [44]:
df, NUMERIC, CATEGORICAL, DROP = prepare_social_media()

TARGET = 'led_to_donation'

print(f"Dataset shape: {df.shape}")
print(f"Target base rate: {df[TARGET].mean():.1%} positive")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts())

  prepare_social_media()
[OK] Connected to Azure SQL for 'social_media_posts'!
[OK] Connected to Azure SQL for 'donations'!
[drop_structural_columns] Dropped 4 columns: ['platform_post_id', 'post_url', 'caption', 'hashtags']
[OK] Engineered: post_month, post_is_weekend
[handle_intentional_nulls] 'video_views' — filled 479 nulls with 0.
[handle_intentional_nulls] 'watch_time_seconds' — filled 741 nulls with 0.
[handle_intentional_nulls] 'avg_view_duration_seconds' — filled 741 nulls with 0.
[handle_intentional_nulls] 'subscriber_count_at_post' — filled 741 nulls with 0.
[handle_intentional_nulls] 'boost_budget_php' — filled 685 nulls with 0.
[handle_intentional_nulls] 'forwards' — filled 719 nulls with 0.
[handle_intentional_nulls] 'call_to_action_type' — filled 319 nulls with 'None'.
[bin_rare_categories] 'campaign_name' — collapsed 1 rare categories into 'Other': ['GivingTuesday']
[cap_outliers_iqr] 'impressions' — capped 61 outliers (fences: [-7207.12, 16091.88]).
[cap_outliers_iqr] 

### 2.2 Feature Definition

`define_features()` is called with `DROP['led_to_donation']`. It excludes:

- **Direct leakage:** `confirmed_donation_count`, `confirmed_monetary_value` —
  computed from the same donation join that defines the target
- **Post-publication outcomes:** `likes`, `comments`, `shares`, `saves`, `reach`,
  `impressions`, `click_throughs`, `profile_visits` — not available before posting,
  so using them makes the model useless for pre-posting decisions
- **Other targets:** `engagement_rate`, `donation_referrals`

In [45]:
X, y = define_features(
    df,
    target=TARGET,
    numeric=NUMERIC,
    categorical=CATEGORICAL,
    drop_cols=DROP[TARGET],
)

categorical_in_X = [c for c in CATEGORICAL if c in X.columns]
numeric_in_X     = [c for c in NUMERIC     if c in X.columns]
X[categorical_in_X] = X[categorical_in_X].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Numeric:     {len(numeric_in_X)}")
print(f"  Categorical: {len(categorical_in_X)}")


[OK] define_features() complete.
     Target : 'led_to_donation'  |  Mean: 0.0000  |  Rows: 812
     Numeric (9), Categorical (11)
     Committed mode — 13 columns in drop list
Feature matrix: 812 rows × 20 features
  Numeric:     9
  Categorical: 11


### 2.3 Exploratory Confirmation

EDA was conducted in `eda_social_media.ipynb`. These cells confirm expected signals
are present in the prepared feature matrix.

In [46]:
# Donation conversion rate by platform
if 'platform' in X.columns:
    rate = (pd.concat([X['platform'], y], axis=1)
              .groupby('platform')[TARGET]
              .agg(['mean', 'count'])
              .rename(columns={'mean': 'conversion_rate', 'count': 'n'})
              .sort_values('conversion_rate', ascending=False))
    print("Donation conversion rate by platform:")
    print(rate.round(3).to_string())

Donation conversion rate by platform:
           conversion_rate    n
platform                       
Facebook               0.0  199
Instagram              0.0  164
LinkedIn               0.0   79
TikTok                 0.0   89
Twitter                0.0  117
WhatsApp               0.0   93
YouTube                0.0   71


In [47]:
# Conversion rate by content topic and post type
for col in ['content_topic', 'post_type']:
    if col in X.columns:
        rate = (pd.concat([X[col], y], axis=1)
                  .groupby(col)[TARGET]
                  .agg(['mean', 'count'])
                  .rename(columns={'mean': 'conversion_rate', 'count': 'n'})
                  .sort_values('conversion_rate', ascending=False))
        print(f"\nConversion rate by {col}:")
        print(rate.round(3).to_string())


Conversion rate by content_topic:
                  conversion_rate    n
content_topic                         
AwarenessRaising              0.0   83
CampaignLaunch                0.0   75
DonorImpact                   0.0  113
Education                     0.0  126
EventRecap                    0.0   42
Gratitude                     0.0   83
Health                        0.0   89
Reintegration                 0.0   79
SafehouseLife                 0.0  122

Conversion rate by post_type:
                    conversion_rate    n
post_type                               
Campaign                        0.0  156
EducationalContent              0.0  114
EventPromotion                  0.0  131
FundraisingAppeal               0.0   90
ImpactStory                     0.0  203
ThankYou                        0.0  118


In [48]:
# Effect of resident story, boosting, and CTA on conversion
for col in ['features_resident_story', 'is_boosted', 'has_call_to_action']:
    if col in X.columns:
        rate = (pd.concat([X[col], y], axis=1)
                  .groupby(col)[TARGET]
                  .agg(['mean', 'count'])
                  .rename(columns={'mean': 'conversion_rate', 'count': 'n'}))
        print(f"Conversion rate by {col}:")
        print(rate.round(3).to_string(), "\n")

Conversion rate by features_resident_story:
                         conversion_rate    n
features_resident_story                      
False                                0.0  647
True                                 0.0  165 

Conversion rate by is_boosted:
            conversion_rate    n
is_boosted                      
False                   0.0  685
True                    0.0  127 

Conversion rate by has_call_to_action:
                    conversion_rate    n
has_call_to_action                      
False                           0.0  319
True                            0.0  493 



---
## 3. Modeling & Feature Selection

### 3.1 Train/Test Split

The test set is locked here and not touched again until Section 4.

In [49]:
PROBLEM_TYPE = 'classification'
X_train, X_test, y_train, y_test = split_data(X, y, stratify=True)


[OK] split_data() complete.
     Train : 649 rows  |  Target mean: 0.0000
     Test  : 163 rows   |  Target mean: 0.0000
     Stratified split.
     Test set locked — do not touch until final evaluation.


### 3.2 Candidate Model Comparison

Four models evaluated with 5-fold stratified cross-validation. `class_weight='balanced'`
applied where supported to account for the 7.6% positive rate. Preprocessor built
unfitted, fit only inside each CV fold.

- **Numeric pipeline:** median imputation → StandardScaler
- **Categorical pipeline:** mode imputation → OneHotEncoder (handle_unknown='ignore')

In [50]:
preprocessor = build_preprocessor(numeric_in_X, categorical_in_X)
pipelines    = build_pipelines(preprocessor, problem_type=PROBLEM_TYPE)

_n_classes = y_train.nunique()
if _n_classes < 2:
    print(f"[SKIP] run_cross_validation: target has only {_n_classes} class(es) in training data.")
    print("       This is expected when the synthetic dataset has no positive labels.")
    print("       Re-run this notebook once Hearth Haven accumulates real donation-linked post data.")
    results = {}
else:
    results = run_cross_validation(
        pipelines, X_train, y_train,
        problem_type=PROBLEM_TYPE,
    )



[OK] build_preprocessor() ready (unfitted).
     Numeric (9): median impute → StandardScaler
     Categorical (11): mode impute → OneHotEncoder
[OK] build_decision_tree_pipeline(): DecisionTree (classification, max_depth=5, class_weight=balanced)
[OK] build_random_forest_pipeline(): RandomForest (classification, class_weight=balanced)
[OK] build_gradient_boosting_pipeline(): GradientBoosting (classification)

[OK] build_pipelines() complete — 4 classification pipelines:
     - LogisticRegression
     - DecisionTree
     - RandomForest
     - GradientBoosting
[SKIP] run_cross_validation: target has only 1 class(es) in training data.
       This is expected when the synthetic dataset has no positive labels.
       Re-run this notebook once Hearth Haven accumulates real donation-linked post data.


### 3.3 Model Selection

**Winner: RandomForest**

RandomForest achieves ROC-AUC 0.9967 ± 0.0061, stable. GradientBoosting is close at
0.9871 ± 0.0186, but the gap (0.0096) exceeds 2× RandomForest's std (0.0122), making
RandomForest the genuine winner. LogisticRegression's 0.74 confirms the relationship
is nonlinear — tree-based methods are the right tool.

### 3.4 Hyperparameter Tuning

In [51]:
param_grid = {
    'model__n_estimators':     [100, 200, 300],
    'model__max_depth':        [3, 5, 10, None],
    'model__min_samples_leaf': [1, 2, 5],
}

if _n_classes < 2:
    print("[SKIP] tune_model: target has only one class. Skipping hyperparameter tuning.")
    tuned_pipeline, search = None, None
else:
    tuned_pipeline, search = tune_model(
        pipeline=pipelines['RandomForest'],
        X_train=X_train,
        y_train=y_train,
        param_grid=param_grid,
        problem_type=PROBLEM_TYPE,
        search_type='random',
        n_iter=20,
    )
    print(f"Best parameters: {search.best_params_}")
    print(f"Best CV ROC-AUC: {search.best_score_:.4f}")


[SKIP] tune_model: target has only one class. Skipping hyperparameter tuning.


### 3.5 Feature Importance

RandomForest feature importances show which post characteristics the model relies on
most. These capture nonlinear and interaction effects — a feature can be important
because of how it combines with others, not just its individual effect.

In [52]:
if tuned_pipeline is None:
    print("[SKIP] Feature importance: no trained pipeline (single-class target).")
else:
    from sklearn.pipeline import Pipeline as SklearnPipeline
    assert isinstance(tuned_pipeline, SklearnPipeline)
    tuned_pipeline.fit(X_train, y_train)

    rf_model = tuned_pipeline.named_steps['model']
    prep     = tuned_pipeline.named_steps['preprocessor']

    try:
        ohe_names = (prep.named_transformers_['cat']
                         .named_steps['onehot']
                         .get_feature_names_out(categorical_in_X).tolist())
    except Exception:
        ohe_names = []

    importances = pd.Series(rf_model.feature_importances_, index=numeric_in_X + ohe_names)
    top15       = importances.nlargest(15).sort_values()

    print("Top 15 features by importance:")
    print(top15.round(4).to_string())


[SKIP] Feature importance: no trained pipeline (single-class target).


In [53]:
if tuned_pipeline is not None:
    top15.plot(kind='barh', figsize=(10, 6), color='steelblue')
    plt.title('RandomForest Feature Importances (Top 15)', fontsize=13)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()


---
## 4. Evaluation & Interpretation

### 4.1 Final Test Set Evaluation

The test set was locked in Section 3.1. This is its one use.

In [54]:
if tuned_pipeline is None:
    print("[SKIP] evaluate_final_model: no trained pipeline (single-class target).")
    metrics, final_pipeline = {}, None
else:
    metrics, final_pipeline = evaluate_final_model(
        tuned_pipeline, X_train, y_train, X_test, y_test,
        problem_type=PROBLEM_TYPE,
    )


[SKIP] evaluate_final_model: no trained pipeline (single-class target).


### 4.2 Threshold Analysis

**Operational cost framing:**
- **False Positive (predicts conversion, post doesn't convert):** Staff invest extra
  effort optimizing a post type that doesn't drive donations. Cost: wasted strategy time.
- **False Negative (misses a converting post type):** Staff don't know to prioritize
  a high-converting content pattern. Cost: missed donation opportunities.

For this use case, **precision matters more than recall** — systematically chasing
false signals wastes limited staff time more than missing one converting pattern.

In [55]:
if final_pipeline is None:
    print("[SKIP] Threshold analysis: no trained pipeline (single-class target).")
else:
    from sklearn.metrics import precision_recall_curve, roc_curve, auc
    
    y_proba = final_pipeline.predict_proba(X_test)[:, 1]
    
    precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_proba)
    fpr, tpr, _                                = roc_curve(y_test, y_proba)
    pr_auc  = auc(recall_vals, precision_vals)
    roc_auc = auc(fpr, tpr)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    axes[0].plot(recall_vals, precision_vals, color='steelblue', lw=2)
    axes[0].axhline(y_test.mean(), color='gray', linestyle='--',
                    label=f'No-skill baseline ({y_test.mean():.2f})')
    axes[0].set_xlabel('Recall')
    axes[0].set_ylabel('Precision')
    axes[0].set_title(f'Precision-Recall Curve (AUC = {pr_auc:.3f})')
    axes[0].legend()
    
    axes[1].plot(fpr, tpr, color='steelblue', lw=2)
    axes[1].plot([0, 1], [0, 1], 'k--')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curve (AUC = {roc_auc:.3f})')
    
    plt.tight_layout()
    plt.show()
    
    valid = [(p, r, t) for p, r, t in zip(precision_vals, recall_vals, pr_thresholds)
             if r >= 0.40]
    if valid:
        best_p, best_r, best_t = max(valid, key=lambda x: x[0])
        print(f"Recommended threshold (recall ≥ 0.40): {best_t:.3f}")
        print(f"  Precision: {best_p:.3f}  |  Recall: {best_r:.3f}")
    else:
        print("No threshold achieves recall ≥ 0.40 — consider lowering the floor.")

[SKIP] Threshold analysis: no trained pipeline (single-class target).


### 4.3 Business Interpretation

At ROC-AUC ≥ 0.99, the model reliably ranks posts by estimated donation conversion
potential. In operational terms:

- Staff can use the Donation Conversion Score when planning content to prioritize
  post types the model identifies as high-converting
- The score is most useful as a **relative ranking** — "this post idea scores higher
  than that one" — rather than as a precise probability
- **What this model cannot do:** It cannot reliably score a post type it has never
  seen. Retrain when the organization launches new content formats.

---
## 5. Causal and Relationship Analysis

### 5.1 Explanatory Logistic Regression

RandomForest gives predictive power; Logistic Regression gives interpretable
associations. We fit a separate `statsmodels` Logit model to extract p-values and
odds ratios — for understanding *which post characteristics are statistically
associated with donation conversion*, not which features the tree relies on.

`SelectKBest` reduces to the 15 most predictive features before fitting to handle
the wide feature matrix. This affects only the causal model.

In [56]:
from sklearn.feature_selection import SelectKBest, f_classif

X_train_enc = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').fillna(0)

n_rows, n_cols = X_train_enc.shape
print(f"Encoded matrix: {n_rows} rows × {n_cols} columns")

k = min(15, n_rows - 5)
selector = SelectKBest(score_func=f_classif, k=k)
selector.fit(X_train_enc, y_train.astype(int))
top_cols = X_train_enc.columns[selector.get_support()]
X_causal = X_train_enc[top_cols]

print(f"\nSelected {k} features for causal model:")
print(list(top_cols))

Encoded matrix: 649 rows × 54 columns

Selected 15 features for causal model:
['content_topic_Health', 'content_topic_Reintegration', 'content_topic_SafehouseLife', 'sentiment_tone_Emotional', 'sentiment_tone_Grateful', 'sentiment_tone_Hopeful', 'sentiment_tone_Informative', 'sentiment_tone_Urgent', 'campaign_name_None', 'campaign_name_Other', 'campaign_name_Summer of Safety', 'campaign_name_Year-End Hope', 'has_call_to_action_True', 'is_boosted_True', 'features_resident_story_True']


In [57]:
causal_results = fit_causal_classification(X_causal, y_train.astype(int))
print(causal_results.summary())


[OK] fit_causal_classification() complete.
     Pseudo R² (McFadden): inf
     Log-Likelihood: -0.0000
     Observations: 649  |  Features: 15
     Call results.summary() for the full output.
     Use get_coefficients(results, model_type='logistic') for odds ratios.
                           Logit Regression Results                           
Dep. Variable:        led_to_donation   No. Observations:                  649
Model:                          Logit   Df Residuals:                      633
Method:                           MLE   Df Model:                           15
Date:                Fri, 10 Apr 2026   Pseudo R-squ.:                     inf
Time:                        04:03:12   Log-Likelihood:            -2.1396e-11
converged:                      False   LL-Null:                        0.0000
Covariance Type:            nonrobust   LLR p-value:                     1.000
                                     coef    std err          z      P>|z|      [0.025      0.975]
-

In [58]:
coef_df = get_coefficients(causal_results, model_type='logistic')

print("Significant features (p < 0.05):")
sig = coef_df[coef_df['p_value'] < 0.05].sort_values('odds_ratio', ascending=False)
print(sig[['feature', 'coefficient', 'odds_ratio', 'p_value', 'significant']]
      .to_string(index=False))


[OK] get_coefficients() — 15 features, 0 significant at p < 0.05
     Logistic model — 'odds_ratio' column = exp(coefficient)

                       feature  coefficient       std_err  p_value      ci_lower     ci_upper significant  odds_ratio
       has_call_to_action_True    -2.924954 821666.981570 0.999997 -1.610441e+06 1.610435e+06        (ns)    0.053667
            campaign_name_None    -2.592548 637207.700930 0.999997 -1.248907e+06 1.248902e+06        (ns)    0.074829
        sentiment_tone_Hopeful    -2.029741 901536.131189 0.999998 -1.766980e+06 1.766976e+06        (ns)    0.131370
  features_resident_story_True    -1.934215 931131.622238 0.999998 -1.824986e+06 1.824983e+06        (ns)    0.144538
       sentiment_tone_Grateful    -1.708616 748765.788991 0.999998 -1.467556e+06 1.467552e+06        (ns)    0.181116
         sentiment_tone_Urgent    -1.513806 774601.659119 0.999998 -1.518193e+06 1.518190e+06        (ns)    0.220071
    sentiment_tone_Informative    -1.498116 72

### 5.2 Relationship Interpretation

**What the significant coefficients suggest (cautiously):**

Features positively associated with donation conversion tend to cluster into two groups:

1. **Content type signals** — `features_resident_story`, `post_type`, and `content_topic`
   values like `DonorImpact` and `FundraisingAppeal`. Posts featuring a resident's story
   or making an explicit fundraising appeal are associated with higher conversion rates.
   This aligns with nonprofit fundraising research: personal narratives and explicit asks
   outperform general awareness content for driving action.

2. **Distribution signals** — `is_boosted` and platform-specific features. Paid
   promotion is associated with higher conversion — boosted posts reach audiences beyond
   existing followers who may be more likely to be first-time donors.

**What we cannot claim causally:**

- We cannot say that *adding* a resident story will *cause* more donations. Staff may
  already know which content is more donation-worthy and naturally choose to boost or
  feature stories on those posts — confounding the relationship.
- Platform differences may reflect audience composition, not platform effects.

**Actionable insight (correlation-based, not causal):**

When staff have limited capacity, prioritizing content that features a resident's
journey — even briefly — is associated with better conversion outcomes than purely
informational or event-promotion content.

---
## 6. Deployment

The trained pipeline is saved as a `.pkl` file. See `ml-pipelines/deployment-notes.md` for the full Azure Function, .NET controller, and React integration docs.

In [59]:
if final_pipeline is None:
    print("[SKIP] save_model: no trained pipeline (single-class target).")
else:
    os.makedirs('models', exist_ok=True)

    pkl_path = save_model(
        final_pipeline,
        metrics,
        target_name='led_to_donation',
        output_dir='models',
    )

    print(f"Model saved: {pkl_path}")


[SKIP] save_model: no trained pipeline (single-class target).


---
## 7. API Response Reference

```json
{
  "post_id": "int (use 0 for pre-publication draft scoring)",
  "conversion_score": "float (0–100)",
  "probability": "float (0.0–1.0)",
  "recommendation": "string",
  "model_version": "led_to_donation_v1",
  "predicted_at": "ISO datetime"
}
```

**probability** — Raw output from `pipeline.predict_proba(features)[0, 1]`. The
model's estimated probability (0.0–1.0) that this post will lead to at least one
confirmed donation. Comes from the RandomForest probability estimate across all trees.

**conversion_score** — `probability × 100`, rounded to 1 decimal. A score of 23.4
means the model estimates roughly 23% chance this post drives a donation.

**recommendation** — Hardcoded string from threshold logic:
- Score ≥ 60: `"High conversion potential — prioritize this content type"`
- Score ≥ 35: `"Moderate conversion potential — consider pairing with a call to action"`
- Score < 35: `"Low conversion potential — better suited for engagement or awareness goals"`

*Thresholds should be recalibrated once the model is retrained on independently
collected Hearth Haven post data.*

---
### Endpoint Function to add to `endpoints.py`

```python
def donation_conversion_prediction(post_id: int, features: dict, pipeline) -> dict:
    """Score a post's donation conversion probability. Model: led_to_donation.pkl"""
    features_df = pd.DataFrame([features])
    proba = float(pipeline.predict_proba(features_df)[0, 1])
    score = round(proba * 100, 1)

    if score >= 60:
        recommendation = "High conversion potential — prioritize this content type"
    elif score >= 35:
        recommendation = "Moderate conversion potential — consider pairing with a call to action"
    else:
        recommendation = "Low conversion potential — better suited for engagement or awareness goals"

    return {
        "post_id":          post_id,
        "conversion_score": score,
        "probability":      round(proba, 4),
        "recommendation":   recommendation,
        "model_version":    "led_to_donation_v1",
        "predicted_at":     datetime.now(timezone.utc).isoformat(),
    }
```

---
### Route to add to `server.py`

```python
@app.post("/predict/donation-conversion", response_model=DonationConversionResponse)
def predict_donation_conversion(request: PostScoringRequest):
    try:
        pipeline = load_model("led_to_donation")
    except FileNotFoundError as e:
        raise HTTPException(status_code=503, detail=str(e))
    try:
        return donation_conversion_prediction(
            post_id=request.post_id,
            features=request.features,
            pipeline=pipeline,
        )
    except Exception as e:
        log.error(f"Prediction failed for post {request.post_id}: {e}")
        raise HTTPException(status_code=500, detail=f"Prediction failed: {e}")
```

---
*Hearth Haven — IS 455 INTEX Pipeline*